<a href="https://colab.research.google.com/github/genfre/Glycosense-Non-Invasive-Glucose-Monitoring/blob/main/hypoglycemia_prediction/notebooks/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Clone your repository so the notebook can see other files in the project
!git clone https://github.com/genfre/Glycosense-Non-Invasive-Glucose-Monitoring.git

# 2. Change the current working directory to the cloned folder
%cd Glycosense-Non-Invasive-Glucose-Monitoring

# 3. Verify you are in the right folder (you should see your repo files)
!ls

Cloning into 'Glycosense-Non-Invasive-Glucose-Monitoring'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 42 (delta 4), reused 38 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 3.11 MiB | 19.10 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Glycosense-Non-Invasive-Glucose-Monitoring/Glycosense-Non-Invasive-Glucose-Monitoring
hypoglycemia_prediction  libs  LICENSE	notebooks  README.md


In [4]:
from huggingface_hub import login
login()
# Paste your token when asked

In [6]:
from huggingface_hub import snapshot_download
import os

local_folder = snapshot_download(repo_id="naomikayegarcia/Glycosense", repo_type="dataset")

print(f"Files downloaded to: {local_folder}")

print("\nHere are the files inside the dataset:")
for root, dirs, files in os.walk(local_folder):
    for file in files:
        print(os.path.join(root, file))

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

28136294.zip:   0%|          | 0.00/25.0G [00:00<?, ?B/s]

✅ Files downloaded to: /root/.cache/huggingface/hub/datasets--naomikayegarcia--Glycosense/snapshots/21922b4d338c67765b53e289045f9dd7b52084b1

Here are the files inside the dataset:
/root/.cache/huggingface/hub/datasets--naomikayegarcia--Glycosense/snapshots/21922b4d338c67765b53e289045f9dd7b52084b1/.gitattributes
/root/.cache/huggingface/hub/datasets--naomikayegarcia--Glycosense/snapshots/21922b4d338c67765b53e289045f9dd7b52084b1/28136294.zip


In [8]:
import os

search_dir = "/root/.cache/huggingface"
zip_path = None

print("🔍 Searching for the zip file...")
for root, dirs, files in os.walk(search_dir):
    for file in files:
        if file.endswith(".zip") and "28136294" in file:
            zip_path = os.path.join(root, file)
            break
    if zip_path: break

if zip_path:
    print(f"✅ Found: {zip_path}")
    output_dir = "/content/extracted_data"
    os.makedirs(output_dir, exist_ok=True)

    print("Starting  unzip...")

    # The '!' runs a shell command.
    # -q means 'quiet' (don't print every filename, or it will crash your browser)
    # -o means 'overwrite' without asking
    !unzip -q -o "$zip_path" -d "$output_dir"

    print(f"Finished! Files are in: {output_dir}")
else:
    print("Could not find the zip file.")

🔍 Searching for the zip file...
✅ Found: /root/.cache/huggingface/hub/datasets--naomikayegarcia--Glycosense/snapshots/21922b4d338c67765b53e289045f9dd7b52084b1/28136294.zip
Starting  unzip...
Finished! Files are in: /content/extracted_data


In [10]:
import zipfile
import os

source_folder = "/content/extracted_data"

final_output = "/content/final_data"
os.makedirs(final_output, exist_ok=True)

print(f"📂 Starting extraction from {source_folder}...")

for filename in os.listdir(source_folder):
    if filename.endswith(".zip"):

        file_path = os.path.join(source_folder, filename)

        folder_name = filename.replace(".zip", "")
        extract_path = os.path.join(final_output, folder_name)
        os.makedirs(extract_path, exist_ok=True)

        try:
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
                print(f"   -> Extracted: {filename}")
        except zipfile.BadZipFile:
            print(f"Warning: {filename} could not be unzipped.")

print(f"\n All done! Your actual data files are in: {final_output}")

📂 Starting extraction from /content/extracted_data...
   -> Extracted: c2s05_raw.zip


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
sys.path.append('..')
import json
import time
import tqdm
import yaml
import torch
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.distributed as dist
from IPython.display import clear_output
from copy import deepcopy
from torch.optim.lr_scheduler import ReduceLROnPlateau
from libs.model import ECG_Inception, PPG_Inception, EDA_LSTM
from torch.utils.data import DataLoader
from libs.dataloader import SeNSEDataset, BalancedBatchSampler

In [ ]:
subject_id = 'c1s01'
version = 'v1'
data_type = 'eda'
config_dir = '../configs'
data_dir = '../data'

In [ ]:
with open(f"{config_dir}/{data_type}.yaml", 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
print(f"Config: {config}")

In [ ]:
data_path = f'{data_dir}/{subject_id}/{data_type}.pkl'
metadata_path = f'{data_dir}/{subject_id}/metadata.json'
data = pd.read_pickle(data_path)
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

In [ ]:
train_data = SeNSEDataset(data, metadata[version]['train'], data_type=data_type, verbose=False)
val_data = SeNSEDataset(data, metadata[version]['val'], data_type=data_type, verbose=False)

train_loader = DataLoader(train_data, batch_size=config['batch_size'], shuffle=False, num_workers=8)
val_loader = DataLoader(val_data, batch_size=config['batch_size'], shuffle=False, num_workers=8)

print(f"Training version {version}")
print("Normal/Hypo ratio: {}".format(train_data.normal_hypo_ratio))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Training on device: {}".format(device))
if data_type == 'ecg':
    model = ECG_Inception(normal_hypo_ratio=train_data.normal_hypo_ratio)
elif data_type == 'ppg':
    model = PPG_Inception(normal_hypo_ratio=train_data.normal_hypo_ratio)
elif data_type == 'eda':
    model = EDA_LSTM(normal_hypo_ratio=train_data.normal_hypo_ratio)
else:
    raise ValueError(f"Unknown data type: {data_type}")
model.to(device)

print("Model size: {}".format(sum(p.numel() for p in model.parameters() if p.requires_grad)))
optimizer = torch.optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=int(config['patience']*0.7), verbose=True)

In [ ]:
# training
best_loss = 1e9
training_losses = []
validating_losses = []
saved_epoch = 0
early_stopping = 0
best_model = None
for epoch in range(config['epochs']):
    model.train()
    training_loss = 0
    train_data.stratified_sampling(batch_size=config['batch_size'])

    start_t = time.time()

    num_batches = len(train_loader)
    for i, (signal_data, hypo_label, glucose, cgm_idx) in enumerate(train_loader):
        optimizer.zero_grad()
        # Forward pass
        if data_type == 'eda':
            phasic, tonic = signal_data
            phasic, tonic, hypo_label, glucose = phasic.float().to(device), tonic.float().to(device), hypo_label.float().to(device), glucose.float().to(device)
            pred_label = model(phasic, tonic)
            num_data = phasic.shape[0]
            if phasic.shape[0] == 1:
                pred_label = pred_label.unsqueeze(0)
        else:
            signal_data, hypo_label, glucose = signal_data.float().to(device), hypo_label.float().to(device), glucose.float().to(device)
            pred_label = model(signal_data)
            num_data = signal_data.shape[0]
            if signal_data.shape[0] == 1:
                pred_label = pred_label.unsqueeze(0)
        loss = model.loss(pred_label, hypo_label, weighted=True)
        loss.backward()
        optimizer.step()
        training_loss += (loss.item() * num_data)


    training_loss = training_loss / len(train_loader.dataset)

    model.eval()
    validating_loss = 0
    with torch.no_grad():
        for signal_data, hypo_label, glucose, cgm_idx in val_loader:
            if data_type == 'eda':
                phasic, tonic = signal_data
                phasic, tonic, hypo_label, glucose = phasic.float().to(device), tonic.float().to(device), hypo_label.float().to(device), glucose.float().to(device)
                pred_label = model(phasic, tonic)
                num_data = phasic.shape[0]
                if phasic.shape[0] == 1:
                    pred_label = pred_label.unsqueeze(0)
            else:
                signal_data, hypo_label, glucose = signal_data.float().to(device), hypo_label.float().to(device), glucose.float().to(device)
                pred_label = model(signal_data)
                num_data = signal_data.shape[0]
                if signal_data.shape[0] == 1:
                    pred_label = pred_label.unsqueeze(0)
            loss = model.loss(pred_label, hypo_label, weighted=True)
            validating_loss += (loss.item() * num_data)

    validating_loss = validating_loss / len(val_loader.dataset)

    print(f"epoch: {epoch}| training_loss: {training_loss:.4f}| validating_loss: {validating_loss:.4f}| time: {time.time()-start_t:.2f}s")

    # Track the best model
    if validating_loss < best_loss:
        best_loss = validating_loss
        best_model = deepcopy(model)
        saved_epoch = epoch
        early_stopping = 0
    else:
        early_stopping += 1

    scheduler.step(validating_loss)


    training_losses.append(training_loss)
    validating_losses.append(validating_loss)

    # Plot the training and validating losses
    clear_output(wait=True)
    plt.figure(figsize=(10, 5))
    plt.plot(training_losses, label='train', color='blue')
    plt.plot(validating_losses, label='val', color='red')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Subject {} - Version {}'.format(subject_id, version))
    plt.tight_layout()
    plt.show()

    # Broadcast early stopping signal
    require_early_stop = early_stopping >= config['patience']
    if require_early_stop:
        print(f"Early stopping at epoch {epoch}")
        break
